<a href="https://colab.research.google.com/github/reshmiee/codemixing_llmrouter/blob/main/merged_fullDataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
from google.colab import files
uploaded = files.upload()   # select all 4 CSVs
print(list(uploaded.keys()))

Saving reasearch_paper_dataset.csv to reasearch_paper_dataset.csv
Saving math_with_hinglish.csv to math_with_hinglish.csv
Saving confirmatory_coding_final.csv to confirmatory_coding_final.csv
Saving ARC_Open_Ended_Hinglish_38 (1).csv to ARC_Open_Ended_Hinglish_38 (1).csv
['reasearch_paper_dataset.csv', 'math_with_hinglish.csv', 'confirmatory_coding_final.csv', 'ARC_Open_Ended_Hinglish_38 (1).csv']


In [4]:
import os
print([f for f in os.listdir('.') if f.endswith('.csv')])

['reasearch_paper_dataset.csv', 'ARC_Open_Ended_Hinglish_38 (1).csv', 'confirmatory_coding_final.csv', 'math_with_hinglish.csv']


In [5]:
import pandas as pd

F_CODING = 'confirmatory_coding_final.csv'
F_MATH   = 'math_with_hinglish.csv'
F_FACT   = 'reasearch_paper_dataset.csv'
F_OPEN   = 'ARC_Open_Ended_Hinglish_38 (1).csv'

def read(f):
    try:
        return pd.read_csv(f, encoding='utf-8')
    except UnicodeDecodeError:           # the factual file is not UTF-8
        return pd.read_csv(f, encoding='cp1252')

cols = ['pair_id', 'task_type', 'difficulty', 'source', 'source_id', 'written_by', 'checked_by',
        'check_status', 'check_note', 'english', 'hinglish', 'hinglish_translated', 'hinglish_devanagari']

cod = read(F_CODING)   # already in the final format, IDs 101-138

f = read(F_FACT).dropna(subset=['id'])          # drops the blank row in the middle
fact = pd.DataFrame({
    'task_type': 'Factual',
    'difficulty': f['difficulty'].values,
    'source': f['difficulty'].map({'Easy': 'ARC-Easy', 'Hard': 'ARC-Challenge'}).values,
    'source_id': f['id'].values,
    'english': f['english'].values,
    'hinglish': f['hinglish'].values})

mth = read(F_MATH)[['task_type', 'difficulty', 'source', 'source_id', 'english', 'hinglish']]

o = read(F_OPEN)
opn = pd.DataFrame({
    'task_type': 'Open-Ended',
    'difficulty': o['Source difficulty'].values,
    'source': 'Written',
    'source_id': '',
    'english': o['Open-ended question (English)'].values,
    'hinglish': o['Hinglish translation'].values})

rest = pd.concat([fact, mth, opn], ignore_index=True)
for c in ['written_by', 'checked_by', 'check_note', 'hinglish_translated', 'hinglish_devanagari']:
    rest[c] = ''
rest['check_status'] = 'PENDING'
start = int(cod['pair_id'].max()) + 1
rest['pair_id'] = range(start, start + len(rest))

master = pd.concat([cod[cols], rest[cols]], ignore_index=True).fillna('')
print(master.groupby(['task_type', 'difficulty']).size())

task_type   difficulty
Coding      Easy          19
            Hard          19
Factual     Easy          19
            Hard          19
Open-Ended  Easy          19
            Hard          19
Reasoning   Easy          19
            Hard          19
dtype: int64


In [6]:
from datasets import load_dataset
import re

def norm(s):
    s = str(s).replace('\u2019', "'").replace('\u2018', "'").replace('\u201c', '"').replace('\u201d', '"')
    return re.sub(r'\s+', ' ', s).strip().lower()

lookup = {}
for cfg in ['ARC-Easy', 'ARC-Challenge']:
    d = load_dataset('allenai/ai2_arc', cfg, split='test').to_pandas()
    for i, q in zip(d['id'], d['question']):
        lookup[norm(q)] = (cfg, i)

n = 0
for idx in master.index[master['task_type'] == 'Factual']:
    h = lookup.get(norm(master.at[idx, 'english']))
    if h:
        master.at[idx, 'source'], master.at[idx, 'source_id'] = h
        n += 1
    else:
        print('no match:', master.at[idx, 'pair_id'])
print('matched', n, 'of 38')

README.md:   0%|          | 0.00/9.00k [00:00<?, ?B/s]

ARC-Easy/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  331kB            

ARC-Easy/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

ARC-Easy/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  346kB            

ARC-Easy/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

...

ARC-Challenge/validation-00000-of-00001.(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/1119 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1172 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/299 [00:00<?, ? examples/s]

no match: 139
no match: 140
no match: 141
no match: 142
no match: 143
no match: 144
no match: 145
no match: 146
no match: 147
no match: 148
no match: 149
no match: 150
no match: 151
no match: 152
no match: 153
no match: 154
no match: 155
no match: 156
no match: 157
no match: 158
no match: 159
no match: 160
no match: 161
no match: 162
no match: 163
no match: 164
no match: 165
no match: 166
no match: 167
no match: 168
no match: 169
no match: 170
no match: 171
no match: 172
no match: 173
no match: 174
no match: 175
no match: 176
matched 0 of 38


In [7]:
from datasets import load_dataset
import re

def norm(s):
    s = str(s).replace('\u2019', "'").replace('\u2018', "'").replace('\u201c', '"').replace('\u201d', '"')
    return re.sub(r'\s+', ' ', s).strip().lower()

lookup = {}   # question text -> (config, id, split)
for cfg in ['ARC-Easy', 'ARC-Challenge']:
    for sp in ['train', 'validation', 'test']:
        d = load_dataset('allenai/ai2_arc', cfg, split=sp).to_pandas()
        for i, q in zip(d['id'], d['question']):
            lookup[norm(q)] = (cfg, i, sp)

found = {}
for idx in master.index[master['task_type'] == 'Factual']:
    h = lookup.get(norm(master.at[idx, 'english']))
    if h:
        found[idx] = h
        master.at[idx, 'source'], master.at[idx, 'source_id'] = h[0], h[1]
    else:
        print('no match:', master.at[idx, 'pair_id'])

print('matched', len(found), 'of 38')
print(pd.Series([h[2] for h in found.values()]).value_counts())   # which split
# does Easy/Hard agree with ARC-Easy/ARC-Challenge?
bad = [master.at[i, 'pair_id'] for i in found
       if {'Easy': 'ARC-Easy', 'Hard': 'ARC-Challenge'}[master.at[i, 'difficulty']] != found[i][0]]
print('difficulty label disagrees with ARC set for pair_ids:', bad)

no match: 163
matched 37 of 38
train    37
Name: count, dtype: int64
difficulty label disagrees with ARC set for pair_ids: []


In [8]:
for q, (cfg, i, sp) in lookup.items():
    if 'digestion process begins' in q:
        print(cfg, i, sp, '|', q)

ARC-Easy Mercury_SC_400329 train | in humans, the digestion process begins in
ARC-Easy CSZ10244 train | the digestion process begins in which of the following?


In [9]:
master.loc[master['pair_id'] == 163, ['source', 'source_id']] = ['ARC-Easy', 'Mercury_SC_400329']
master.loc[master['pair_id'] == 163, 'check_note'] = 'English reworded from ARC stem'
print(master.loc[master['pair_id'] == 163, ['source', 'source_id', 'english']].to_string())

      source          source_id                                                             english
62  ARC-Easy  Mercury_SC_400329  In humans, the digestion process begins in which part of the body?


In [10]:
import re

# collapse double spaces in English
master['english'] = master['english'].str.replace(r' {2,}', ' ', regex=True)

# pair 197: LaTeX-style quotes
for col in ['english', 'hinglish']:
    master.loc[master['pair_id'] == 197, col] = (
        master.loc[master['pair_id'] == 197, col].str.replace("`SIXTEEN'", "'SIXTEEN'", regex=False))

# pair 145, 149, 163 Hinglish typos
master.loc[master['pair_id'] == 145, 'hinglish'] = (
    "Rain forests mein tree species kisi bhi aur biome se zyada hoti hain. "
    "Phir bhi scientists ne paaya hai ki forest floor ki mitti mein nutrients kaafi kam hote hain. "
    "Iska kya karan ho sakta hai?")
master.loc[master['pair_id'] == 149, 'hinglish'] = (
    master.loc[master['pair_id'] == 149, 'hinglish'].str.replace('mukable', 'muqable', regex=False))
master.loc[master['pair_id'] == 163, 'hinglish'] = (
    master.loc[master['pair_id'] == 163, 'hinglish'].str.replace('nsaanon', 'Insaanon', regex=False))

In [12]:
print('mojibake:', master.astype(str).apply(lambda c: c.str.contains('Ã|Â|â€', regex=True)).sum().sum())
print('BOM check after save below')

mojibake: 0
BOM check after save below


In [14]:
print('rows:', len(master), '| duplicate ids:', master['pair_id'].duplicated().sum())
print('blank english:', (master['english'].str.strip() == '').sum(),
      '| blank hinglish:', (master['hinglish'].str.strip() == '').sum())
print('broken characters:', master.astype(str).apply(lambda c: c.str.contains('\ufffd')).sum().sum())

master.to_csv('confirmatoryv1_draft.csv', index=False, encoding='utf-8-sig')
print(open('confirmatoryv1_draft.csv', 'rb').read(3))
from google.colab import files
files.download('confirmatory_v1_draft.csv')

rows: 152 | duplicate ids: 0
blank english: 0 | blank hinglish: 0
broken characters: 0
b'\xef\xbb\xbf'


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>